# SwimHPE Training

Fine-tunes a YOLO-Pose model on the SwimXYZ dataset.

**Sections:**
1. **Mini run** — sample 16 train/16 val images, train for a quick sanity check
2. **Augmentation explorer** — visualise augmented training samples with configurable params
3. **Full training** — train on the complete SwimXYZ train set

### YOLO26-Pose Architecture Overview

**1. Backbone (Layers 0–8): The Feature Extractor**
* **Components:** `Conv` layers, `C3k2` (efficient bottlenecks), `SPPF` (spatial pooling), and `C2PSA` (spatial attention).
* **Role:** Progressively shrinks the image resolution while extracting deep, complex visual patterns. 

**2. Neck (Layers 9-22): The Feature Aggregator**
* **Components:** `Upsample`, `Concat`, and `C3k2` blocks (forming a PANet).
* **Role:** Blends high-resolution spatial details from early layers with deep semantic meaning from later layers.

**3. Head (Layer 23): The Predictor**
* **Components:** End-to-End `Pose` Head.
* **Role:** Outputs final bounding boxes and keypoints. It uses Residual Log-Likelihood Estimation (RLE) for sub-pixel joint precision and completely eliminates the need for Non-Maximum Suppression (NMS) post-processing.


In [26]:
# Install any packages not pre-installed on this runtime (safe to re-run; skipped if already present)
import subprocess, sys

_REQUIRED = ["ultralytics"]
for _pkg in _REQUIRED:
    try:
        __import__(_pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", _pkg, "-q"], check=True)

In [27]:
import os
import sys
import random
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from ultralytics import YOLO
from ultralytics.utils import SETTINGS

# Must be set before MPS/OpenMP are initialised
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

In [28]:
import sys

_IN_COLAB = 'google.colab' in sys.modules

if _IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # ---- Set this to your Drive root (dataset/ must be directly inside) ----
    _ROOT = Path('/content/drive/MyDrive')
    # ------------------------------------------------------------------------
else:
    _ROOT = Path.cwd()
    if _ROOT.name == 'training':
        _ROOT = _ROOT.parent

print(f'Project root: {_ROOT}')
print(f'Running on Colab: {_IN_COLAB}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive
Running on Colab: True


In [29]:
# Disable Ultralytics analytics callbacks that can block between epochs
SETTINGS['sync'] = False
try:
    from ultralytics.utils.events import events as _ul_events
    _ul_events.enabled = False
except Exception:
    pass

In [30]:
def train_and_save(
    model="yolo11n-pose.pt",
    epochs=2,
    device="cpu",
    freeze=9,
    data=None,
    plots=True,
    val=True,
    save=True,
    save_period=0,
    final_val=False,
):
    """
    Train the model and save the final weights.

    Args:
        model: Ultralytics model identifier or path to weights file.
            Passed directly to YOLO(); auto-downloaded from hub if not found locally.
            Examples: "yolo11n-pose.pt", "yolo11s-pose.pt", "path/to/custom.pt".
        epochs: Number of training epochs.
        device: Device to train on (e.g. "cpu", "0", "mps").
        freeze: Number of top-level backbone/neck blocks to freeze
            (0 = train all, passed directly to model.train()).
        data: Path to the dataset YAML config. Defaults to configs/swimXYZ.yaml.
        plots: Whether to save training/val metric plots to the run directory.
        val: Whether to run validation after each epoch.
        save: Whether to save last.pt/best.pt checkpoints.
        save_period: Save a checkpoint every N epochs (0 = disabled).
        final_val: Whether to run a single validation pass after training.

    Returns:
        Tuple of (best_weights_path, run_dir).
    """
    data = data or str(_ROOT / 'configs/swimXYZ.yaml')

    # save_period requires save=True — Ultralytics only calls save_model() when save=True,
    # and the epochN.pt logic lives inside save_model().
    if save_period > 0:
        save = True

    print(f"Loading model: {model}")
    yolo = YOLO(model)

    # Seed the trainer's callback lists as empty so integration callbacks start from scratch.
    yolo.callbacks["on_fit_epoch_end"] = []
    yolo.callbacks["on_train_end"] = []

    def _suppress_network_callbacks(trainer) -> None:
        # Fires at on_train_start, after add_integration_callbacks() has already appended
        # hub/platform callbacks to the trainer's own dict.
        for key in ("on_fit_epoch_end", "on_train_end", "teardown"):
            trainer.callbacks[key] = []

    yolo.callbacks["on_train_start"].append(_suppress_network_callbacks)

    yolo.train(
        data=data,
        epochs=epochs,
        imgsz=640,
        device=device,
        workers=0,
        plots=plots,
        val=val,
        save=save,
        save_period=save_period,
        freeze=freeze,
        project=str(_ROOT / 'runs'),
        name="train",
    )

    run_dir = Path(yolo.trainer.save_dir)
    best_pt = run_dir / 'weights' / 'best.pt'
    print("\n✅ Training complete.")
    print(f"   Run artifacts : {run_dir}")
    print(f"   Best weights  : {best_pt}")

    if final_val:
        print("\nRunning final validation...")
        yolo.val(data=data, device=device)

    return str(best_pt), run_dir

In [31]:
def setup_mini_dataset(n_train=16, n_val=16, seed=42):
    """Create dataset/mini/ with images and labels, return config path."""
    rng = random.Random(seed)

    src_img_train = _ROOT / 'dataset' / 'images' / 'train'
    src_img_val   = _ROOT / 'dataset' / 'images' / 'val'
    src_lbl_train = _ROOT / 'dataset' / 'labels' / 'train'
    src_lbl_val   = _ROOT / 'dataset' / 'labels' / 'val'

    mini_root = _ROOT / 'dataset' / 'mini'

    if mini_root.exists():
        shutil.rmtree(mini_root)

    for split in ('train', 'val'):
        (mini_root / 'images' / split).mkdir(parents=True)
        (mini_root / 'labels' / split).mkdir(parents=True)

    def _link(src: Path, dst: Path) -> None:
        """Symlink locally; copy on Colab (Drive FUSE doesn't support symlinks)."""
        if _IN_COLAB:
            shutil.copy2(src, dst)
        else:
            dst.symlink_to(src.resolve())

    def _symlink_split(img_src, lbl_src, img_dst, lbl_dst, n):
        all_images = sorted(img_src.glob('*'))
        chosen = rng.sample(all_images, min(n, len(all_images)))
        for img in chosen:
            _link(img, img_dst / img.name)
            lbl = (lbl_src / img.stem).with_suffix('.txt')
            if lbl.exists():
                _link(lbl, lbl_dst / lbl.name)

    _symlink_split(src_img_train, src_lbl_train,
                   mini_root / 'images' / 'train',
                   mini_root / 'labels' / 'train',
                   n_train)
    _symlink_split(src_img_val, src_lbl_val,
                   mini_root / 'images' / 'val',
                   mini_root / 'labels' / 'val',
                   n_val)

    config_path = _ROOT / 'configs' / 'mini.yaml'
    config_path.parent.mkdir(parents=True, exist_ok=True)
    config_path.write_text(
        "# Auto-generated by mini_run.ipynb — do not commit\n"
        f"path: {mini_root.resolve()}\n"
        "train: images/train\n"
        "val: images/val\n"
        "kpt_shape: [13, 3]\n"
        "flip_idx: [1, 0, 3, 2, 5, 4, 7, 6, 9, 8, 11, 10, 12]\n"
        "names:\n"
        "    0: person\n"
    )

    n_train_actual = len(list((mini_root / 'images' / 'train').iterdir()))
    n_val_actual   = len(list((mini_root / 'images' / 'val').iterdir()))
    print(f"Mini dataset ready: {n_train_actual} train / {n_val_actual} val images")
    print(f"Config: {config_path}")
    return config_path

In [32]:
# Pairs of (title, train_col, val_col) — missing columns are silently skipped
_LOSS_PANELS = [
    ("Box loss",       "train/box_loss",  "val/box_loss"),
    ("Pose loss",      "train/pose_loss", "val/pose_loss"),
    ("Keypoint obj.",  "train/kobj_loss", "val/kobj_loss"),
    ("Class loss",     "train/cls_loss",  "val/cls_loss"),
    ("DFL loss",       "train/dfl_loss",  "val/dfl_loss"),
]

_METRIC_PANELS = [
    ("Box mAP50",       "metrics/mAP50(B)",    None),
    ("Box mAP50-95",    "metrics/mAP50-95(B)", None),
    ("Pose mAP50",      "metrics/mAP50(P)",    None),
    ("Pose mAP50-95",   "metrics/mAP50-95(P)", None),
    ("Precision (box)", "metrics/precision(B)", None),
    ("Recall (box)",    "metrics/recall(B)",    None),
]


def plot_training_results(run_dir):
    """Read results.csv from run_dir and display loss + metric curves."""
    csv_path = run_dir / 'results.csv'
    if not csv_path.exists():
        print(f"⚠️  results.csv not found at {csv_path}. Skipping plots.")
        return

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    epochs = df['epoch'] if 'epoch' in df.columns else range(len(df))

    available_loss    = [(t, tc, vc) for t, tc, vc in _LOSS_PANELS if tc in df.columns]
    available_metrics = [(t, mc, _) for t, mc, _ in _METRIC_PANELS if mc in df.columns]
    all_panels = available_loss + available_metrics

    if not all_panels:
        print("⚠️  No recognised columns found in results.csv. Available columns:")
        print("   ", list(df.columns))
        return

    ncols = 3
    nrows = (len(all_panels) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = axes.flatten() if nrows > 1 else [axes] if ncols == 1 else list(axes)
    fig.suptitle(f"Training results — {run_dir.name}", fontsize=13, fontweight='bold')

    for ax, (title, train_col, val_col) in zip(axes, all_panels):
        ax.plot(epochs, df[train_col], label='train', linewidth=1.5)
        if val_col and val_col in df.columns:
            ax.plot(epochs, df[val_col], label='val', linewidth=1.5, linestyle='--')
        ax.set_title(title)
        ax.set_xlabel('epoch')
        ax.legend()
        ax.grid(True, alpha=0.3)

    for ax in axes[len(all_panels):]:
        ax.set_visible(False)

    plt.tight_layout()
    out_path = run_dir / 'loss_curves.png'
    plt.savefig(out_path, dpi=150)
    print(f"✅ Loss/metric curves saved to {out_path}")

    results_png = run_dir / 'results.png'
    if results_png.exists():
        print(f"   Ultralytics results.png: {results_png}")

    plt.show()

In [33]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

mini_config = setup_mini_dataset(n_train=16, n_val=16, seed=42)

_weights_path, run_dir = train_and_save(
    data=str(mini_config),
    epochs=200,
    device=device,
    freeze=9,
    plots=False,      # skip per-epoch Ultralytics PNG generation
    val=False,        # skip per-epoch validation; custom plots still work from results.csv
    save=False,       # no per-epoch checkpoints; final weights saved by train_and_save
    final_val=True,   # single validation pass on trained model after training
    model="yolo26m-pose.pt",
    save_period=50
)

plot_training_results(run_dir)

Using device: cuda
Mini dataset ready: 16 train / 16 val images
Config: /content/drive/MyDrive/configs/mini.yaml
Loading model: yolo26m-pose.pt
Ultralytics 8.4.18 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/configs/mini.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=9, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m-pose.pt, momentum=

KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## Augmentation Explorer

Visualise what augmented training samples look like **before** committing to a set of hyperparameters.

Edit the `AUG` dictionary in the cell below, then run it to see a grid of samples produced by the exact same pipeline used during training (mosaic, HSV jitter, perspective, fliplr with the correct keypoint `flip_idx`, erasing, etc.).

| Param | Default | Range | Effect |
|---|---|---|---|
| `hsv_h` | 0.015 | 0–1 | Hue shift |
| `hsv_s` | 0.70 | 0–1 | Saturation jitter |
| `hsv_v` | 0.40 | 0–1 | Brightness jitter |
| `degrees` | 15.0 | 0–180 | Rotation ± degrees |
| `translate` | 0.10 | 0–1 | H/V translation fraction |
| `scale` | 0.50 | 0–1 | Scale gain (0.5 → 0.5×–1.5×) |
| `shear` | 0.0 | 0–180 | Shear ± degrees |
| `perspective` | 0.0 | 0–0.001 | Perspective warp |
| `flipud` | 0.0 | 0–1 | Vertical flip probability |
| `fliplr` | 0.5 | 0–1 | Horizontal flip probability |
| `bgr` | 0.0 | 0–1 | BGR channel swap probability |
| `mosaic` | 1.0 | 0–1 | 4-image mosaic probability |
| `mixup` | 0.0 | 0–1 | MixUp blend probability |
| `erasing` | 0.4 | 0–1 | Random region erasure probability |

In [ ]:
import numpy as np
from types import SimpleNamespace
from ultralytics.data.build import build_yolo_dataset
from ultralytics.data.utils import check_det_dataset

# ── Edit these augmentation params, then re-run the cell ──────────────────
AUG = dict(
    hsv_h        = 0.015,   # Hue shift (0–1)
    hsv_s        = 0.70,    # Saturation jitter (0–1)
    hsv_v        = 0.40,    # Brightness jitter (0–1)
    degrees      = 15.0,    # Rotation ± degrees (0–180)
    translate    = 0.10,    # H/V translation as fraction (0–1)
    scale        = 0.50,    # Scale gain (0.5 → 0.5×–1.5×)
    shear        = 0.0,     # Shear ± degrees (0–180)
    perspective  = 0.0,     # Perspective warp (0–0.001, keep small)
    flipud       = 0.0,     # Vertical flip probability (0–1)
    fliplr       = 0.5,     # Horizontal flip probability (0–1)
    bgr          = 0.0,     # BGR channel-swap probability (0–1)
    mosaic       = 1.0,     # 4-image mosaic probability (0–1)
    mixup        = 0.0,     # MixUp blend probability (0–1)
    erasing      = 0.4,     # Random region erasure probability (0–1)
)

N_IMAGES = 8   # number of augmented samples to show
N_COLS   = 4
# ──────────────────────────────────────────────────────────────────────────

data_cfg  = str(_ROOT / 'configs' / 'swimXYZ.yaml')
data_dict = check_det_dataset(data_cfg)

cfg = SimpleNamespace(
    task="pose", mode="train",
    imgsz=640, rect=False, cache=False,
    single_cls=False, fraction=1.0, pad=0.5,
    overlap_mask=True, mask_ratio=4,
    close_mosaic=0,
    copy_paste=0.0, copy_paste_mode="flip",
    cutmix=0.0,
    **AUG,
)

img_path = str(Path(data_dict["path"]) / data_dict["train"])
dataset  = build_yolo_dataset(cfg, img_path, batch=N_IMAGES, data=data_dict, mode="train")

n_rows = (N_IMAGES + N_COLS - 1) // N_COLS
fig, axes = plt.subplots(n_rows, N_COLS, figsize=(5 * N_COLS, 4 * n_rows))
axes = axes.flatten() if n_rows > 1 else list(axes)
fig.suptitle("Augmented training samples (swimXYZ)", fontsize=13, fontweight="bold")

for i, ax in enumerate(axes):
    if i >= N_IMAGES:
        ax.set_visible(False)
        continue
    sample = dataset[i % len(dataset)]
    img = sample["img"]
    if isinstance(img, torch.Tensor):
        img = img.numpy()
    img = img.transpose(1, 2, 0)  # CHW → HWC
    if img.dtype != np.uint8:
        img = (img * 255).clip(0, 255).astype(np.uint8)
    ax.imshow(img[..., ::-1])    # BGR → RGB
    ax.set_title(f"sample {i}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

---
## Full Training

Trains on the complete SwimXYZ train set using `configs/swimXYZ.yaml`.

**Before running:** tune the `AUG` dict in the Augmentation Explorer cell above and copy the values you want into the `model.train()` call below.

**Tip:** lower `FREEZE` (e.g. `5`) or set to `0` to train all layers if the mini run shows underfitting.

In [ ]:
# ── Full training config ───────────────────────────────────────────────────
MODEL        = 'yolo11n-pose.pt'   # base model to fine-tune
EPOCHS       = 100
FREEZE       = 9     # number of backbone/neck blocks to freeze from the start (0 = train all)
CLOSE_MOSAIC = 10    # disable mosaic for the final N epochs

# Paste your tuned AUG values from the explorer above (or leave as defaults)
FULL_AUG = dict(
    hsv_h        = 0.015,
    hsv_s        = 0.70,
    hsv_v        = 0.40,
    degrees      = 15.0,
    translate    = 0.10,
    scale        = 0.50,
    shear        = 0.0,
    perspective  = 0.0,
    flipud       = 0.0,
    fliplr       = 0.5,
    bgr          = 0.0,
    mosaic       = 1.0,
    mixup        = 0.0,
    erasing      = 0.4,
)
# ──────────────────────────────────────────────────────────────────────────

data_cfg = str(_ROOT / 'configs' / 'swimXYZ.yaml')

print(f"Loading model: {MODEL}")
model = YOLO(MODEL)

# Suppress hub/sync callbacks (same as train_and_save)
SETTINGS['sync'] = False
model.callbacks["on_fit_epoch_end"] = []
model.callbacks["on_train_end"] = []

def _suppress_network_callbacks(trainer):
    for key in ("on_fit_epoch_end", "on_train_end", "teardown"):
        trainer.callbacks[key] = []

model.callbacks["on_train_start"].append(_suppress_network_callbacks)

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {device}")

model.train(
    data=data_cfg,
    epochs=EPOCHS,
    imgsz=640,
    device=device,
    workers=4,
    plots=True,
    val=True,
    save=True,
    freeze=FREEZE,
    close_mosaic=CLOSE_MOSAIC,
    project=str(_ROOT / 'runs'),
    name="full_train",
    **FULL_AUG,
)

run_dir = Path(model.trainer.save_dir)
best_pt = run_dir / 'weights' / 'best.pt'
print(f"\n✅ Full training complete.")
print(f"   Run artifacts : {run_dir}")
print(f"   Best weights  : {best_pt}")

plot_training_results(run_dir)